# Test LangChain MCP Server via HTTP

This notebook demonstrates how to query the LangChain documentation MCP server
using raw HTTP requests (JSON-RPC 2.0 over HTTP).

In [ ]:
import requests
import json

In [ ]:
MCP_URL = "https://docs.langchain.com/mcp"

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream"
}

session_id = None  # Will be set after initialize

## Step 1: Initialize the MCP session

This is the handshake. The server returns its capabilities and a session ID.

In [ ]:
init_payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {
            "name": "notebook-test-client",
            "version": "1.0.0"
        }
    }
}

resp = requests.post(MCP_URL, headers=HEADERS, json=init_payload)
print(f"Status: {resp.status_code}")
print(f"Response Headers: {dict(resp.headers)}")
print()

# Capture session ID if returned
session_id = resp.headers.get("Mcp-Session-Id") or resp.headers.get("mcp-session-id")
print(f"Session ID: {session_id}")
print()

# Parse response (handle SSE or plain JSON)
if "text/event-stream" in resp.headers.get("content-type", ""):
    for line in resp.text.strip().split("\n"):
        if line.startswith("data: "):
            data = json.loads(line[6:])
            print(json.dumps(data, indent=2))
else:
    data = resp.json()
    print(json.dumps(data, indent=2))

## Step 2: Send initialized notification

Tell the server we're ready. This is a notification (no `id` field, no response expected).

In [ ]:
headers_with_session = {**HEADERS}
if session_id:
    headers_with_session["Mcp-Session-Id"] = session_id

notif_payload = {
    "jsonrpc": "2.0",
    "method": "notifications/initialized"
}

resp = requests.post(MCP_URL, headers=headers_with_session, json=notif_payload)
print(f"Status: {resp.status_code}")
print(f"Response: {resp.text[:500] if resp.text else '(empty)'}")

## Step 3: List available tools

Ask the server what tools it exposes.

In [ ]:
list_tools_payload = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/list",
    "params": {}
}

resp = requests.post(MCP_URL, headers=headers_with_session, json=list_tools_payload)
print(f"Status: {resp.status_code}")
print()

if "text/event-stream" in resp.headers.get("content-type", ""):
    for line in resp.text.strip().split("\n"):
        if line.startswith("data: "):
            data = json.loads(line[6:])
            print(json.dumps(data, indent=2))
else:
    data = resp.json()
    print(json.dumps(data, indent=2))

## Step 4: Search the docs

Call the `search_docs_by_lang_chain` tool to search for documentation.

In [ ]:
search_payload = {
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "search_docs_by_lang_chain",
        "arguments": {
            "query": "how to create an agent with LangGraph"
        }
    }
}

resp = requests.post(MCP_URL, headers=headers_with_session, json=search_payload)
print(f"Status: {resp.status_code}")
print()

if "text/event-stream" in resp.headers.get("content-type", ""):
    for line in resp.text.strip().split("\n"):
        if line.startswith("data: "):
            data = json.loads(line[6:])
            print(json.dumps(data, indent=2))
else:
    data = resp.json()
    print(json.dumps(data, indent=2))

## Step 5: Query the virtual filesystem

Use the `query_docs_filesystem_docs_by_lang_chain` tool to explore docs structure or read pages.

In [ ]:
# List the top-level directory structure
filesystem_payload = {
    "jsonrpc": "2.0",
    "id": 4,
    "method": "tools/call",
    "params": {
        "name": "query_docs_filesystem_docs_by_lang_chain",
        "arguments": {
            "command": "tree / -L 2"
        }
    }
}

resp = requests.post(MCP_URL, headers=headers_with_session, json=filesystem_payload)
print(f"Status: {resp.status_code}")
print()

if "text/event-stream" in resp.headers.get("content-type", ""):
    for line in resp.text.strip().split("\n"):
        if line.startswith("data: "):
            data = json.loads(line[6:])
            print(json.dumps(data, indent=2))
else:
    data = resp.json()
    print(json.dumps(data, indent=2))

In [ ]:
# Read a specific documentation page
read_page_payload = {
    "jsonrpc": "2.0",
    "id": 5,
    "method": "tools/call",
    "params": {
        "name": "query_docs_filesystem_docs_by_lang_chain",
        "arguments": {
            "command": "head -50 /quickstart.mdx"
        }
    }
}

resp = requests.post(MCP_URL, headers=headers_with_session, json=read_page_payload)
print(f"Status: {resp.status_code}")
print()

if "text/event-stream" in resp.headers.get("content-type", ""):
    for line in resp.text.strip().split("\n"):
        if line.startswith("data: "):
            data = json.loads(line[6:])
            print(json.dumps(data, indent=2))
else:
    data = resp.json()
    print(json.dumps(data, indent=2))

## Step 6: Search for a keyword in the filesystem

Use ripgrep (`rg`) to find files mentioning a specific term.

In [ ]:
rg_payload = {
    "jsonrpc": "2.0",
    "id": 6,
    "method": "tools/call",
    "params": {
        "name": "query_docs_filesystem_docs_by_lang_chain",
        "arguments": {
            "command": "rg -il \"retrieval\" /"
        }
    }
}

resp = requests.post(MCP_URL, headers=headers_with_session, json=rg_payload)
print(f"Status: {resp.status_code}")
print()

if "text/event-stream" in resp.headers.get("content-type", ""):
    for line in resp.text.strip().split("\n"):
        if line.startswith("data: "):
            data = json.loads(line[6:])
            print(json.dumps(data, indent=2))
else:
    data = resp.json()
    print(json.dumps(data, indent=2))

---

## Helper: Reusable function

A convenience function to make any MCP tool call in one line.

In [ ]:
def call_mcp_tool(tool_name: str, arguments: dict, request_id: int = 99):
    """Call any tool on the LangChain MCP server and return parsed result."""
    payload = {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": "tools/call",
        "params": {
            "name": tool_name,
            "arguments": arguments
        }
    }
    resp = requests.post(MCP_URL, headers=headers_with_session, json=payload)
    
    if "text/event-stream" in resp.headers.get("content-type", ""):
        results = []
        for line in resp.text.strip().split("\n"):
            if line.startswith("data: "):
                results.append(json.loads(line[6:]))
        return results[-1] if len(results) == 1 else results
    else:
        return resp.json()


# Example usage:
result = call_mcp_tool("search_docs_by_lang_chain", {"query": "RAG tutorial"})
print(json.dumps(result, indent=2))